# Credit Default Model (Stacking & Ensembling) - 100% Retrain Strategy

ไฟล์นี้ถูกปรับแต่งให้เป็นสูตร **Hybrid Strategy**:
1. **ใช้ CV (Cross-Validation)** เพื่อหาค่าเฉลี่ยของจำนวนต้นไม้ (Iterations) ที่ดีที่สุด (Best Iterations)
2. **Retrain 100%** โดยใช้ Parameter เดิม และยัดข้อมูล Train ทั้งหมดลงไปรันตามจำนวนต้นไม้เฉลี่ยที่ได้จากข้อ 1 (ปิด Early Stopping)
3. สร้างไฟล์ Probability สำหรับส่งประกวดโดยไม่สน Threshold ตัดคลาสแล้ว

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except ImportError:          # รันนอก Jupyter
    display = print

# ---------- ไฟล์และชื่อรอบ ----------
DATA_DIR = Path(r"C:\Users\ice55\Downloads\Aihack Thailand 2026\data")   # <-- แก้ path ตรงนี้
RUN_NAME = "final_retrain100"           # ชื่อรอบ
TARGET   = "12-Month Default Flag"

# ---------- สวิตช์ ----------
USE_CATBOOST   = True        # เปิดใช้ CatBoost
USE_GENDER     = False       # ปิดไว้
USE_ID_FEATURE = False       # ปิดไว้

# ---------- Cross-validation ----------
SEED    = 42
N_FOLDS = 10

if USE_CATBOOST:
    from catboost import CatBoostClassifier

## 1. ข้อมูล
โหลดไฟล์ และตรวจสอบ

In [ ]:
train    = pd.read_csv(DATA_DIR / "train_dataset.csv")
test     = pd.read_csv(DATA_DIR / "test_without_gt.csv")
template = pd.read_csv(DATA_DIR / "submission_template_for_test.csv")

assert train["ID"].is_unique and test["ID"].is_unique
assert set(template["ID"]) == set(test["ID"]), "ID ใน template ไม่ตรงกับ test"

print(f"train {train.shape} | test {test.shape} | template {template.shape}")
print(f"default rate (train) = {train[TARGET].mean():.4f}")

## 2. Features
สร้าง ratio และ frequency features

In [ ]:
CAT_COLS = [
    "Application Channel", "Marital Status", "Occupation Category", "Industry Category",
    "Employment Type Category", "Insurance Type", "Business Entity Type",
    "Residence Type", "Residence Ownership Status",
]
if USE_GENDER:
    CAT_COLS.append("Gender")

PHONE_COLS = ["Home Phone Number Available", "Mobile Phone Number Available",
              "Employer Phone Number Available", "Driver Licence Number Available"]

def safe_div(a, b):
    b = b.astype(float)
    return a / b.where(b > 0)

def make_features(df):
    d = df.copy()
    income = d["Declared Annual Income"]
    debt   = d["Declared Borrowing Amount"]
    limit  = d["Desired Credit Limit"]
    n_dep  = d["Number of Dependents"]

    d["borrow_to_income"]   = safe_div(debt, income)
    d["limit_to_income"]    = safe_div(limit, income)
    d["exposure_to_income"] = safe_div(debt + limit, income)
    d["limit_to_borrow"]    = safe_div(limit, debt)
    d["avg_loan"]           = safe_div(debt, d["Declared Number of Borrowings"])
    d["rent_to_income"]       = safe_div(d["Rent Burden"], income)
    d["income_per_dependent"] = income / (n_dep + 1)
    d["children_share"]       = safe_div(d["Number of Dependent Children"], n_dep)
    d["residence_ratio"] = safe_div(d["Residence Period (in months)"], d["Age (in months)"])
    d["phone_count"]     = d[PHONE_COLS].sum(axis=1)
    return d

def add_freq_features(d):
    pairs = {
        "occ_x_ind":     ("Occupation Category", "Industry Category"),
        "size_x_entity": ("Company Size Category", "Business Entity Type"),
        "occ_x_emp":     ("Occupation Category", "Employment Type Category"),
    }
    for name, (a, b) in pairs.items():
        key = d[a].astype(str) + "_" + d[b].astype(str)
        d[name + "_freq"] = key.map(key.value_counts(normalize=True))
    return d

full = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True)
full = add_freq_features(make_features(full))
if USE_ID_FEATURE:
    full["id_prefix"] = full["ID"] // 10000
for c in CAT_COLS:
    full[c] = full[c].astype("category")

drop_cols    = ["ID"] + ([] if USE_GENDER else ["Gender"])
RAW_FEATURES = [c for c in train.columns if c not in drop_cols + [TARGET]]
ENG_FEATURES = [c for c in full.columns if c not in train.columns]
ALL_FEATURES = RAW_FEATURES + ENG_FEATURES

X_all  = full.iloc[:len(train)].reset_index(drop=True)
X_test = full.iloc[len(train):].reset_index(drop=True)
y      = train[TARGET].values

print(f"{len(RAW_FEATURES)} raw + {len(ENG_FEATURES)} engineered = {len(ALL_FEATURES)} features")

## 3. หา Best Parameters ผ่าน Cross-validation
ใช้ CV เพื่อดูค่าเฉลี่ยของจำนวนต้นไม้ที่ดีที่สุดในแต่ละโมเดล (Best Iterations)

In [ ]:
folds      = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_all, y))
scoreboard = []

# ตัวแปรสำหรับเก็บค่าเฉลี่ยต้นไม้ที่ดีที่สุด
best_iters_lgb = []
best_iters_cat = []

def run_cv_lgb(features, name):
    oof, fold_aucs = np.zeros(len(X_all)), []
    for tr, va in folds:
        model = lgb.LGBMClassifier(**LGB_PARAMS)
        model.fit(X_all.iloc[tr][features], y[tr], eval_set=[(X_all.iloc[va][features], y[va])],
                  eval_metric="auc", callbacks=[lgb.early_stopping(200, verbose=False)])
        
        p_val = model.predict_proba(X_all.iloc[va][features])[:, 1]
        oof[va] = p_val
        fold_aucs.append(roc_auc_score(y[va], p_val))
        best_iters_lgb.append(model.best_iteration_)
        
    row = dict(experiment=name, oof_auc=roc_auc_score(y, oof), fold_mean=np.mean(fold_aucs))
    scoreboard.append(row)
    print(f"{name:<28} OOF AUC {row['oof_auc']:.4f} | fold {row['fold_mean']:.4f}")
    return oof

# ---------- LightGBM ----------
LGB_PARAMS = dict(
    n_estimators=5000, 
    learning_rate=0.02,
    num_leaves=15,
    min_child_samples=200,
    subsample=0.8, 
    subsample_freq=1, 
    colsample_bytree=0.7,
    reg_lambda=10.0, 
    cat_smooth=50, 
    min_data_per_group=200,
    random_state=SEED, 
    n_jobs=-1, 
    verbose=-1,
)

# ---------- CatBoost ----------
def _cat_to_str(X):
    X = X.copy()
    for c in CAT_COLS:
        if c in X.columns:
            X[c] = X[c].astype(str)
    return X

def run_cv_cat(features, name):
    oof, fold_aucs = np.zeros(len(X_all)), []
    cat_cols = [c for c in CAT_COLS if c in X_all.columns]
    X_str = _cat_to_str(X_all)
    
    for tr, va in folds:
        model = CatBoostClassifier(iterations=3000, learning_rate=0.03, depth=6, l2_leaf_reg=10,
                                   eval_metric="AUC", random_seed=SEED, early_stopping_rounds=200,
                                   cat_features=cat_cols, verbose=0, thread_count=-1)
        model.fit(X_str.iloc[tr][features], y[tr], eval_set=(X_str.iloc[va][features], y[va]), use_best_model=True)
        
        p_val = model.predict_proba(X_str.iloc[va][features])[:, 1]
        oof[va] = p_val
        fold_aucs.append(roc_auc_score(y[va], p_val))
        best_iters_cat.append(model.get_best_iteration())
        
    row = dict(experiment=name, oof_auc=roc_auc_score(y, oof), fold_mean=np.mean(fold_aucs))
    scoreboard.append(row)
    print(f"{name:<28} OOF AUC {row['oof_auc']:.4f} | fold {row['fold_mean']:.4f}")
    return oof

print("--- 1. รัน CV เพื่อหา Best Iterations ---")
print("Training LGBM (CV)...")
oof_lgb = run_cv_lgb(ALL_FEATURES, "LGBM (CV)")

if USE_CATBOOST:
    print("Training CatBoost (CV)...")
    oof_cat = run_cv_cat(ALL_FEATURES, "CatBoost (CV)")

# คำนวณค่าเฉลี่ยต้นไม้
avg_iter_lgb = int(np.mean(best_iters_lgb)) if best_iters_lgb else 0
avg_iter_cat = int(np.mean(best_iters_cat)) if best_iters_cat else 0

print(f"\n✅ หาค่าเฉลี่ยต้นไม้เสร็จสิ้น")
print(f"LightGBM Best Avg Iterations: {avg_iter_lgb}")
print(f"CatBoost Best Avg Iterations: {avg_iter_cat}")

## 4. Retrain โมเดลด้วยข้อมูล Train 100%
นำค่าเฉลี่ยต้นไม้ที่ได้จากข้อ 3 (avg_iter_lgb, avg_iter_cat) มาตั้งค่า แล้วเทรนบน X_all แบบไม่มี Early Stopping แล้วทำนายออกไปที่ Test Data ทันที

In [ ]:
print("--- 2. Retrain 100% Data ---")
final_test_preds = {}

# ----- Retrain LightGBM -----
print("Retraining LightGBM on 100% Data...")
# ก๊อปปี้พารามิเตอร์เดิม แต่เปลี่ยนจำนวนต้นไม้ให้เท่ากับค่าเฉลี่ยที่หามาได้
retrain_lgb_params = LGB_PARAMS.copy()
retrain_lgb_params["n_estimators"] = avg_iter_lgb

model_lgb_full = lgb.LGBMClassifier(**retrain_lgb_params)
model_lgb_full.fit(X_all[ALL_FEATURES], y) # ไม่มี eval_set, ไม่มี early_stopping
pred_lgb_full = model_lgb_full.predict_proba(X_test[ALL_FEATURES])[:, 1]
final_test_preds["LGBM"] = pred_lgb_full

# ----- Retrain CatBoost -----
if USE_CATBOOST:
    print("Retraining CatBoost on 100% Data...")
    X_str_all = _cat_to_str(X_all)
    X_str_test = _cat_to_str(X_test)
    cat_cols = [c for c in CAT_COLS if c in X_str_all.columns]
    
    # เปลี่ยน iterations เป็นค่าเฉลี่ยที่ได้จาก CV
    model_cat_full = CatBoostClassifier(iterations=avg_iter_cat, learning_rate=0.03, depth=6, l2_leaf_reg=10,
                                        random_seed=SEED, cat_features=cat_cols, verbose=0, thread_count=-1)
    model_cat_full.fit(X_str_all[ALL_FEATURES], y) # ไม่มี eval_set
    pred_cat_full = model_cat_full.predict_proba(X_str_test[ALL_FEATURES])[:, 1]
    final_test_preds["CatBoost"] = pred_cat_full

print("✅ Retrain 100% เสร็จสิ้น")

## 5. รวมผล Ensembling Strategies (สำหรับชุด Test)
เนื่องจากเรา Retrain 100% เราจึงรวมผลได้เฉพาะกับค่า Prediction ของฝั่ง Test Data (ไม่ได้มี OOF ให้พล็อตรวมอีกแล้ว)
ในรอบนี้ เราจะใช้วิธี Weighted Average เป็นตัวส่งหลัก (เพราะง่ายและเสถียรที่สุดเมื่อไม่มี OOF)

In [ ]:
print("--- 3. Ensembling on Retrained Predictions ---")

# 1. Simple Blending
pred_blend = np.mean(list(final_test_preds.values()), axis=0)

# 2. Weighted Average (0.65 LGBM : 0.35 CatBoost)
w_lgb, w_cat = 0.65, 0.35
if "CatBoost" in final_test_preds:
    pred_weight = (final_test_preds["LGBM"] * w_lgb) + (final_test_preds["CatBoost"] * w_cat)
else:
    pred_weight = final_test_preds["LGBM"]

# 3. Stacking: เมื่อ Retrain 100% จะไม่สามารถทำ Stacking ได้โดยตรงแบบเดิม เพราะเราไม่มีค่า OOF ของแต่ละโมเดลมา fit meta-model แล้ว
# ดังนั้นสำหรับการ Retrain 100% วิธี Weighted Average มักจะปลอดภัยที่สุดครับ
pred_final = pred_weight

print("✅ จับรวมผลลัพธ์บน Test Data เสร็จสิ้น")

## 6. สร้างไฟล์ Submit (Probability Only)
สร้างไฟล์ส่งออกแบบมีแค่ความน่าจะเป็น 100%

In [ ]:
# =========================================================
# สร้างไฟล์ Submit สำหรับ Test Data (เฉพาะความน่าจะเป็น)
# =========================================================
def make_submission(values, path):
    sub = pd.DataFrame({"ID": test["ID"].values, TARGET: values})
    sub = template[["ID"]].merge(sub, on="ID", how="left")
    sub.to_csv(path, index=False)
    return sub

proba_file = f"submission_{RUN_NAME}.csv"

# บันทึกไฟล์
sub_proba = make_submission(pred_final, proba_file)

print(f"--- 4. สรุปการเตรียมไฟล์ Submit ---")
print(f"ไฟล์สำหรับส่ง (Probability Only): {proba_file}")
print("นำไฟล์นี้ไปอัปโหลดส่งในระบบเพื่อดูคะแนน Private / Public แบบ 100% ได้เลยครับ")